In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

sentences = [
    "double balloon enteroscopy",
    "DBE procedure",
    "MPH program in public health education",
]
emb = model.encode(sentences, normalize_embeddings=True)

print(emb.shape)
print(emb[0][:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(3, 384)
[ 0.00517013 -0.03241267  0.04125675  0.02034077 -0.0627156  -0.00492294
  0.02344549  0.03694337 -0.03980056 -0.04320025]


In [2]:
import numpy as np

sim = emb @ emb.T
print(np.round(sim, 3))

[[1.    0.575 0.492]
 [0.575 1.    0.516]
 [0.492 0.516 1.   ]]


In [3]:
import json, torch

with open("../data/corpus.json") as f:
    corpus = json.load(f)
with open("../data/questions.json") as f:
    questions = json.load(f)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
print("device:", device)

doc_emb = model.encode(
    [doc["text"] for doc in corpus],
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
np.save("../data/doc_emb.npy", doc_emb)
print(doc_emb.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

device: mps


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

(6000, 384)


In [4]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def dense_search(query, corpus, k=5):
    q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    scores = doc_emb @ q
    top = np.argsort(-scores)[:k]
    return [corpus[i]["doc_id"] for i in top]

def evaluate(search_fn, n=300):
    hits1, hits5 = 0, 0
    for q in questions[:n]:
        found = search_fn(q["question"], corpus, k=5)
        hits1 += found[0] == q["gold_doc_id"]
        hits5 += q["gold_doc_id"] in found
    return hits1 / n, hits5 / n

r1, r5 = evaluate(dense_search)
print(f"dense   Recall@1={r1:.3f}  Recall@5={r5:.3f}")

dense   Recall@1=0.953  Recall@5=0.977


In [5]:
import sys
sys.path.append("../src")
from bm25 import BM25

bm25 = BM25([doc["text"] for doc in corpus])

def bm25_search(query, corpus, k=5):
    return [corpus[i]["doc_id"] for score, i in bm25.search(query, k)]

only_dense, only_bm25, both_miss = [], [], []
for q in questions[:300]:
    d_ok = dense_search(q["question"], corpus, k=1)[0] == q["gold_doc_id"]
    b_ok = bm25_search(q["question"], corpus, k=1)[0] == q["gold_doc_id"]
    if d_ok and not b_ok:
        only_dense.append(q)
    elif b_ok and not d_ok:
        only_bm25.append(q)
    elif not d_ok and not b_ok:
        both_miss.append(q)

print("dense прав, BM25 нет:", len(only_dense))
print("BM25 прав, dense нет:", len(only_bm25))
print("оба ошиблись:        ", len(both_miss))

dense прав, BM25 нет: 20
BM25 прав, dense нет: 6
оба ошиблись:         8


In [6]:
for q in only_bm25:
    print("-", q["question"])

- Can predilatation in transcatheter aortic valve implantation be omitted?
- Should circumcision be performed in childhood?
- Does the type of tibial component affect mechanical alignment in unicompartmental knee replacement?
- Can patients be anticoagulated after intracerebral hemorrhage?
- Are we seeing the effects of public awareness campaigns?
- Prescriptions as a proxy for asthma in children: a good choice?


In [7]:
from collections import defaultdict

def rrf(rankings, k=5, rrf_k=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (rrf_k + rank)
    best = sorted(scores.items(), key=lambda x: -x[1])
    return [doc_id for doc_id, s in best[:k]]

def hybrid_search(query, corpus, k=5, pool=50):
    return rrf([bm25_search(query, corpus, k=pool),
                dense_search(query, corpus, k=pool)], k=k)

r1, r5 = evaluate(hybrid_search)
print(f"hybrid  Recall@1={r1:.3f}  Recall@5={r5:.3f}")

hybrid  Recall@1=0.953  Recall@5=0.973


In [8]:
gained, lost = [], []
for q in questions[:300]:
    d_ok = dense_search(q["question"], corpus, k=1)[0] == q["gold_doc_id"]
    h_ok = hybrid_search(q["question"], corpus, k=1)[0] == q["gold_doc_id"]
    if h_ok and not d_ok:
        gained.append(q)
    elif d_ok and not h_ok:
        lost.append(q)

print("hybrid исправил ошибки dense:", len(gained))
print("hybrid сломал то, что dense решал:", len(lost))

hybrid исправил ошибки dense: 5
hybrid сломал то, что dense решал: 5


In [9]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
id2text = {doc["doc_id"]: doc["text"] for doc in corpus}

def rerank_search(query, corpus, k=5, pool=20):
    candidates = hybrid_search(query, corpus, k=pool)
    pairs = [(query, id2text[doc_id]) for doc_id in candidates]
    scores = reranker.predict(pairs)
    order = np.argsort(-scores)[:k]
    return [candidates[i] for i in order]

r1, r5 = evaluate(rerank_search)
print(f"rerank  Recall@1={r1:.3f}  Recall@5={r5:.3f}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

rerank  Recall@1=0.950  Recall@5=0.977
